# Heat Pump Workflow

Compare a base case against a candidate heat-pump integration scenario using
OpenPinch's public helper surface.

This notebook focuses on targeting and process integration. The key question is
whether a candidate temperature lift reduces the right utility demands and
improves the thermal picture.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from OpenPinch import PinchProblem
from OpenPinch.resources import copy_sample_case

workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
case_path = copy_sample_case(
    "heat_pump_targeting.json",
    work_dir / "heat_pump_targeting.json",
)
problem = PinchProblem(problem_filepath=case_path)
base_summary = problem.summary_frame()
base_summary.loc[base_summary["Target"] == "Plant/Direct Integration"]


## Candidate integration scenario

The scenario below applies a condenser at 170 degC delivering 500 kW and an
evaporator at 90 degC lifting 400 kW. The difference is a simple stand-in for
compressor-power demand.

The main output is the before-and-after utility comparison, not the isolated
cycle performance.


In [ ]:
evaluation = problem.evaluate_heat_pump_integration(
    {
        "zone": "Plant",
        "condenser_temperature": 170.0,
        "condenser_duty": 500.0,
        "evaporator_temperature": 90.0,
        "evaporator_duty": 400.0,
    },
    target_name="Plant/Direct Integration",
)
evaluation.comparison_frame


## Reading the comparison

For this scenario, the hot utility target should fall materially, the cold
utility target should also improve, and the heat recovery should increase.
Those are the first indicators that the temperature lift is helping rather than
simply shifting duty around the plant.


In [ ]:
integrated_gcc = evaluation.integrated_problem.plot_grand_composite_curve(
    zone_name="Plant/Direct Integration"
)
[
    evaluation.comparison.model_dump(),
    integrated_gcc.layout.title.text,
]
